In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "meta-llama/llama-3.3-70b-instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_path = kagglehub.dataset_download("nikhileswarkomati/suicide-watch")
data = pd.read_csv(dataset_path + "/Suicide_Detection.csv").sample(frac=0.3, random_state=0)
data["label"] = data["class"].astype("category").cat.codes
texts = data["text"].astype(str).tolist()
labels = data["label"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake

    preds = []
    probs = []
    indices = []

    for i, text in enumerate(batch):

        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )

        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=1,
                logprobs=True,
                top_logprobs=10
            )

            content = r.choices[0].message.content

            if content is None:
                formatted_mistake += 1
                continue

            out = content.strip()

            if out == "1":
                parsed = 1
            elif out == "0":
                parsed = 0
            else:
                print("Unexpected output:", repr(out))
                formatted_mistake += 1
                continue
            token_info = r.choices[0].logprobs.content[0]

            logprob_0 = None
            logprob_1 = None

            for candidate in token_info.top_logprobs:
                token = candidate.token.strip()

                if token == "0":
                    logprob_0 = candidate.logprob

                elif token == "1":
                    logprob_1 = candidate.logprob

            if logprob_0 is None or logprob_1 is None:
                print("0 or 1 missing from top_logprobs")
                formatted_mistake += 1
                continue

            p0 = np.exp(logprob_0)
            p1 = np.exp(logprob_1)
            prob_1 = p1 / (p0 + p1)

            preds.append(parsed)
            probs.append(prob_1)
            indices.append(i)

        except Exception as e:
            print("API error:", e)
            formatted_mistake += 1
            continue

    return preds, probs, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
probs = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_probs, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    probs.append(np.array(batch_probs))
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
probs = np.concatenate(probs)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, probs)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

100%|██████████| 60.6M/60.6M [00:01<00:00, 48.2MB/s]

Extracting files...



  0%|          | 0/63 [00:00<?, ?it/s]

API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'c

  2%|▏         | 1/63 [00:24<25:16, 24.46s/it]

...

API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'c

 16%|█▌        | 10/63 [02:46<12:36, 14.27s/it]

API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'
API error: 'NoneType' object has no attribute 'content'


 16%|█▌        | 10/63 [02:49<14:58, 16.95s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "meta-llama/llama-3.3-70b-instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
dataset_path = kagglehub.dataset_download("rvarun11/suicidal-ideation-reddit-dataset")
data = pd.read_csv(dataset_path + "/suicidal_ideation_reddit_annotated.csv")
data["text"] = data["usertext"].astype(str)
texts = data["text"].tolist()
labels = data["label"].astype(int).tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

100%|██████████| 3.79M/3.79M [00:01<00:00, 3.43MB/s]

Extracting files...



 29%|██▊       | 18/63 [07:24<20:08, 26.86s/it]


Unexpected output: 'There is'


 56%|█████▌    | 35/63 [14:57<12:26, 26.65s/it]

...


EMPTY RESPONSE
finish_reason: stop


 86%|████████▌ | 54/63 [25:08<05:40, 37.83s/it]


EMPTY RESPONSE
finish_reason: stop


100%|██████████| 63/63 [29:14<00:00, 27.84s/it]

Inference time: 1754.163000207
Accuracy: 0.8873
False positive: 186
False negative: 38
Precision: 0.8475409836065574
Recall: 0.9645522388059702
F1: 0.9022687609075044
ROC-AUC: 0.8807477351235091
formatted mistake:  0.006036217303822937
12
1988


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "meta-llama/llama-3.3-70b-instruct"
import kagglehub
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mapping = {
    "non-suicide": 0,
    "suicide": 1
}
dataset_path = kagglehub.dataset_download("neelakshigupta/suicide-detection")
data = pd.read_csv(dataset_path + "/file1.csv")
data["text"] = data["text"]
data["Suicide"] = data["class"].map(mapping)
data = data.dropna(subset = ["text", "class"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

100%|██████████| 1.45M/1.45M [00:00<00:00, 1.69MB/s]

Extracting files...



  6%|▋         | 4/63 [02:10<34:20, 34.93s/it]


EMPTY RESPONSE
finish_reason: stop


 10%|▉         | 6/63 [03:01<28:03, 29.53s/it]

...


Unexpected output: 'I cannot'


 97%|█████████▋| 61/63 [33:47<01:10, 35.04s/it]


Unexpected output: '!!'


100%|██████████| 63/63 [35:00<00:00, 33.35s/it]

Inference time: 2100.972917465
Accuracy: 0.7937
False positive: 396
False negative: 12
Precision: 0.7068837897853442
Recall: 0.9875904860392968
F1: 0.8239861949956859
ROC-AUC: 0.7979495456902715
formatted mistake:  0.011122345803842264
22
1978


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "meta-llama/llama-3.3-70b-instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
mapping = {
    "Not Suicide post": 0,
    "Potential Suicide post ": 1
}
dataset_path = kagglehub.dataset_download("aunanya875/suicidal-tweet-detection-dataset")
data = pd.read_csv(dataset_path + "/Suicide_Ideation_Dataset(Twitter-based).csv")
data["text"] = data["Tweet"]
data["Suicide"] = data["Suicide"].map(mapping)
data = data.dropna(subset=["Tweet", "Suicide"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

100%|██████████| 88.3k/88.3k [00:00<00:00, 416kB/s]

Extracting files...



 43%|████▎     | 24/56 [12:27<15:55, 29.87s/it]


Unexpected output: "I'm"


 66%|██████▌   | 37/56 [18:04<08:13, 25.95s/it]

...


Unexpected output: '!!'

EMPTY RESPONSE
finish_reason: stop


 93%|█████████▎| 52/56 [25:46<02:00, 30.22s/it]


EMPTY RESPONSE
finish_reason: stop


100%|██████████| 56/56 [27:58<00:00, 29.98s/it]

Inference time: 1678.7584412409997
Accuracy: 0.8658
False positive: 48
False negative: 190
Precision: 0.90625
Recall: 0.709480122324159
F1: 0.7958833619210978
ROC-AUC: 0.8333114897335081
formatted mistake:  0.0062006764374295375
11
1774


# Grok

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "x-ai/grok-4.3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_path = kagglehub.dataset_download("nikhileswarkomati/suicide-watch")
data = pd.read_csv(dataset_path + "/Suicide_Detection.csv").sample(frac=0.3, random_state=0)
data["label"] = data["class"].astype("category").cat.codes
texts = data["text"].astype(str).tolist()
labels = data["label"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicide-watch' dataset.


100%|██████████| 63/63 [18:57<00:00, 18.06s/it]

Inference time: 1137.653012775998
Accuracy: 0.9290
False positive: 74
False negative: 68
Precision: 0.924643584521385
Recall: 0.930327868852459
F1: 0.9274770173646578
ROC-AUC: 0.9290311219262295
formatted mistake:  0.0
0
2000


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "x-ai/grok-4.3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
dataset_path = kagglehub.dataset_download("rvarun11/suicidal-ideation-reddit-dataset")
data = pd.read_csv(dataset_path + "/suicidal_ideation_reddit_annotated.csv")
data["text"] = data["usertext"].astype(str)
texts = data["text"].tolist()
labels = data["label"].astype(int).tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicidal-ideation-reddit-dataset' dataset.


100%|██████████| 63/63 [19:25<00:00, 18.50s/it]

Inference time: 1165.6284557319996
Accuracy: 0.8730
False positive: 153
False negative: 101
Precision: 0.8626570915619389
Recall: 0.9048964218455744
F1: 0.8832720588235294
ROC-AUC: 0.8708917077244931
formatted mistake:  0.0
0
2000


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "x-ai/grok-4.3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
mapping = {
    "Not Suicide post": 0,
    "Potential Suicide post ": 1
}
dataset_path = kagglehub.dataset_download("aunanya875/suicidal-tweet-detection-dataset")
data = pd.read_csv(dataset_path + "/Suicide_Ideation_Dataset(Twitter-based).csv")
data["text"] = data["Tweet"]
data["Suicide"] = data["Suicide"].map(mapping)
data = data.dropna(subset=["Tweet", "Suicide"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicidal-tweet-detection-dataset' dataset.


100%|██████████| 56/56 [17:06<00:00, 18.33s/it]

Inference time: 1026.5269813689993
Accuracy: 0.8319
False positive: 15
False negative: 285
Precision: 0.961439588688946
Recall: 0.56752655538695
F1: 0.7137404580152672
ROC-AUC: 0.7771025316899226
formatted mistake:  0.0
0
1785


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "REDACTED"
model_name = "x-ai/grok-4.3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
mapping = {
    "non-suicide": 0,
    "suicide": 1
}
dataset_path = kagglehub.dataset_download("neelakshigupta/suicide-detection")
data = pd.read_csv(dataset_path + "/file1.csv")
data["text"] = data["text"]
data["Suicide"] = data["class"].map(mapping)
data = data.dropna(subset = ["text", "class"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort="none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicide-detection' dataset.


100%|██████████| 63/63 [19:14<00:00, 18.32s/it]

Inference time: 1154.4656555229994
Accuracy: 0.8685
False positive: 232
False negative: 31
Precision: 0.8082644628099174
Recall: 0.9692765113974232
F1: 0.8814781433077963
ROC-AUC: 0.8675847743667238
formatted mistake:  0.0
0
2000
